# NeuroGuard — MRI classifier training v2 (OASIS-3, binary, 5-fold CV)

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Two ways to provide data (pick ONE):
1. **Preprocessed slices** — upload a zip of `processed_v2/` (slices + manifest_v2.csv) to Drive as `processed_v2.zip`.
2. **Raw NIfTI + download more** — provide OASIS XNAT credentials below to download the full 518-subject list, then slices are generated here.

In [ ]:
!git clone --depth 1 https://github.com/akashjacob2005-rgb/final-year-project.git /content/repo
!pip -q install nibabel onnx onnxscript onnxruntime requests

In [ ]:
# OPTION 1: preprocessed slices from Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/data && unzip -q -o /content/drive/MyDrive/processed_v2.zip -d /content/data
import glob
DATA_DIR = glob.glob('/content/data/**/manifest_v2.csv', recursive=True)[0].rsplit('/', 1)[0]
print('data dir:', DATA_DIR)

In [ ]:
# OPTION 2 (instead of option 1): download the full list, then make slices.
# Also upload processed/session_labels.csv + download_list.csv to Drive first:
#   /content/drive/MyDrive/oasis_meta/{session_labels.csv,download_list.csv}
import os
os.environ['OASIS_USER'] = ''   # <- fill in
os.environ['OASIS_PASS'] = ''   # <- fill in
!mkdir -p /content/oasis/processed && cp /content/drive/MyDrive/oasis_meta/*.csv /content/oasis/processed/
!python /content/repo/ml/mri/download_oasis.py --data-root /content/oasis
!python /content/repo/ml/mri/make_slices.py --data-root /content/oasis
DATA_DIR = '/content/oasis/processed_v2'

In [ ]:
!python /content/repo/ml/mri/train_mri.py --data-dir "$DATA_DIR" --out-dir /content/artifacts --label-mode binary --folds 5 --epochs 30

In [ ]:
import json
m = json.load(open('/content/artifacts/mri_metrics.json'))
print(json.dumps(m['cv'], indent=2))
from google.colab import files
files.download('/content/artifacts/mri_model.onnx')
files.download('/content/artifacts/mri_metrics.json')
# NOTE: torch may also write mri_model.onnx.data (external weights) — download it too if present:
import os
if os.path.exists('/content/artifacts/mri_model.onnx.data'):
    files.download('/content/artifacts/mri_model.onnx.data')

Copy `mri_model.onnx` (+ `.data` if present) and `mri_metrics.json` into the repo's `ml/artifacts/` to deploy.